# 🚦 AI Traffic Intelligence & Road Safety System

YOLOv8 + ByteTrack traffic analytics prototype.

This notebook uses an already-mounted YOLO-ready UA-DETRAC dataset in Kaggle. It does **not** download the dataset.


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q ultralytics opencv-python matplotlib pandas pyyaml


In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/notebooks/jprafi/ua-detrac/yolo_uadetrac')

TRAIN_IMAGES = DATASET_ROOT / 'train' / 'images'
TRAIN_LABELS = DATASET_ROOT / 'train' / 'labels'
VAL_IMAGES = DATASET_ROOT / 'val' / 'images'
VAL_LABELS = DATASET_ROOT / 'val' / 'labels'

print('Dataset exists:', DATASET_ROOT.exists())


In [ ]:
def count_files(folder, extensions):
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions)

image_ext = {'.jpg', '.jpeg', '.png'}

train_images = count_files(TRAIN_IMAGES, image_ext)
train_labels = count_files(TRAIN_LABELS, {'.txt'})
val_images = count_files(VAL_IMAGES, image_ext)
val_labels = count_files(VAL_LABELS, {'.txt'})

print('TRAIN images:', train_images)
print('TRAIN labels:', train_labels)
print('VAL images:', val_images)
print('VAL labels:', val_labels)


In [ ]:
import yaml

DATA_YAML = Path('/kaggle/working/traffic.yaml')

config = {
    'path': str(DATASET_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'nc': 4,
    'names': {
        0: 'car',
        1: 'van',
        2: 'bus',
        3: 'others'
    }
}

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(DATA_YAML.read_text())


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

image_files = [p for p in TRAIN_IMAGES.iterdir() if p.suffix.lower() in image_ext]
image_path = random.choice(image_files)

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 7))
plt.imshow(image)
plt.title(image_path.name)
plt.axis('off')
plt.show()


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print('YOLO model loaded successfully')


In [ ]:
# Train the custom traffic detector.
# If CUDA runs out of memory, change batch=16 to batch=8.

results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=8,
    project='/kaggle/working/traffic_training',
    name='yolov8n_traffic',
    pretrained=True,
    plots=True
)


In [ ]:
# Load the trained model before tracking.
model = YOLO('/kaggle/working/traffic_training/yolov8n_traffic/weights/best.pt')
print('Trained model loaded')


In [ ]:
VIDEO_PATH = '/kaggle/input/datasets/mohanakash08/okkkkkkkk/WhatsApp Video 2026-09-23 at 21.55.41.mp4'

results = model.track(
    source=VIDEO_PATH,
    tracker='bytetrack.yaml',
    conf=0.25,
    persist=True,
    save=True,
    project='/kaggle/working/traffic_tracking',
    name='bytetrack_test'
)

print('Tracking completed!')


In [ ]:
from IPython.display import Video, display

TRACKED_VIDEO = '/kaggle/working/traffic_tracking/bytetrack_test/WhatsApp Video 2026-09-23 at 21.55.41.avi'
display(Video(TRACKED_VIDEO, embed=True))


In [ ]:
# Basic vehicle counting using a horizontal counting line.
from collections import defaultdict

LINE_Y = 300
counted_ids = set()
vehicle_counts = defaultdict(int)

results = model.track(
    source=TRACKED_VIDEO,
    tracker='bytetrack.yaml',
    conf=0.25,
    stream=True,
    verbose=False
)

for result in results:
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    classes = result.boxes.cls.cpu().numpy().astype(int)

    for box, track_id, cls_id in zip(boxes, ids, classes):
        x1, y1, x2, y2 = box
        center_y = (y1 + y2) / 2

        if center_y > LINE_Y and track_id not in counted_ids:
            counted_ids.add(track_id)
            vehicle_counts[model.names[cls_id]] += 1

print('Vehicle Counting Completed')
print('--------------------------------')
print('Total vehicles:', len(counted_ids))
for vehicle, count in vehicle_counts.items():
    print(f'{vehicle}: {count}')


In [ ]:
# Experimental IN/OUT counting. Kept for reference; line placement depends on camera geometry.
LINE_Y = 300
previous_y = {}
in_ids = set()
out_ids = set()
in_counts = defaultdict(int)
out_counts = defaultdict(int)

results = model.track(
    source=TRACKED_VIDEO,
    tracker='bytetrack.yaml',
    persist=True,
    conf=0.25,
    stream=True,
    verbose=False
)

for result in results:
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    classes = result.boxes.cls.cpu().numpy().astype(int)

    for box, track_id, cls_id in zip(boxes, ids, classes):
        x1, y1, x2, y2 = box
        center_y = (y1 + y2) / 2

        if track_id in previous_y:
            old_y = previous_y[track_id]

            if old_y < LINE_Y <= center_y and track_id not in in_ids and track_id not in out_ids:
                in_ids.add(track_id)
                in_counts[model.names[cls_id]] += 1

            elif old_y > LINE_Y >= center_y and track_id not in out_ids and track_id not in in_ids:
                out_ids.add(track_id)
                out_counts[model.names[cls_id]] += 1

        previous_y[track_id] = center_y

print('IN / OUT COUNTING COMPLETED')
print('----------------------------')
print('IN :', len(in_ids))
print('OUT:', len(out_ids))
print('TOTAL:', len(in_ids) + len(out_ids))


In [ ]:
# Relative speed estimation (pixels/frame, not km/h).
import math

previous_position = {}
speed_history = defaultdict(list)

results = model.track(
    source=TRACKED_VIDEO,
    tracker='bytetrack.yaml',
    conf=0.25,
    stream=True,
    verbose=False
)

for result in results:
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)

    for box, track_id in zip(boxes, ids):
        x1, y1, x2, y2 = box
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        if track_id in previous_position:
            old_x, old_y = previous_position[track_id]
            distance = math.sqrt((center_x - old_x) ** 2 + (center_y - old_y) ** 2)
            speed_history[track_id].append(distance)

        previous_position[track_id] = (center_x, center_y)

print('Speed estimation completed')
print('---------------------------')
print('Vehicles tracked:', len(speed_history))
for track_id, speeds in list(speed_history.items())[:10]:
    if speeds:
        print(f'Vehicle {track_id}: {sum(speeds) / len(speeds):.2f} pixels/frame')


In [ ]:
# 5-second traffic presence trend.
FPS = 30
WINDOW_SECONDS = 5
WINDOW_FRAMES = FPS * WINDOW_SECONDS
window_vehicle_ids = defaultdict(set)

results = model.track(
    source=TRACKED_VIDEO,
    tracker='bytetrack.yaml',
    conf=0.25,
    stream=True,
    verbose=False
)

frame_number = 0
for result in results:
    if result.boxes is not None and result.boxes.id is not None:
        ids = result.boxes.id.cpu().numpy().astype(int)
        window_number = frame_number // WINDOW_FRAMES
        for track_id in ids:
            window_vehicle_ids[window_number].add(track_id)
    frame_number += 1

print('5-SECOND TRAFFIC PRESENCE')
print('--------------------------')
for window, ids in sorted(window_vehicle_ids.items()):
    start = window * WINDOW_SECONDS
    end = start + WINDOW_SECONDS
    print(f'{start:02d}-{end:02d}s : {len(ids)} vehicles')


In [ ]:
# Direction analysis from tracked center-point movement.
previous_position = {}
movement = defaultdict(lambda: [0.0, 0.0])

results = model.track(
    source=TRACKED_VIDEO,
    tracker='bytetrack.yaml',
    conf=0.25,
    stream=True,
    verbose=False
)

for result in results:
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)

    for box, track_id in zip(boxes, ids):
        x1, y1, x2, y2 = box
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        if track_id in previous_position:
            old_x, old_y = previous_position[track_id]
            dx = center_x - old_x
            dy = center_y - old_y
            if abs(dx) > 1 or abs(dy) > 1:
                movement[track_id][0] += dx
                movement[track_id][1] += dy

        previous_position[track_id] = (center_x, center_y)

print('DIRECTION ANALYSIS')
print('------------------')
for track_id, (dx, dy) in sorted(movement.items()):
    if abs(dx) > abs(dy):
        direction = 'RIGHT' if dx > 0 else 'LEFT'
    else:
        direction = 'DOWN' if dy > 0 else 'UP'
    print(f'Vehicle {track_id}: {direction} (dx={dx:.1f}, dy={dy:.1f})')


In [ ]:
# Direction summary using trajectory angle.
direction_counts = Counter()

for track_id, (dx, dy) in movement.items():
    distance = math.sqrt(dx**2 + dy**2)

    if distance < 10:
        direction = 'STATIONARY'
    else:
        angle = math.degrees(math.atan2(-dy, dx))
        if -22.5 <= angle < 22.5:
            direction = 'RIGHT'
        elif 22.5 <= angle < 67.5:
            direction = 'UP-RIGHT'
        elif 67.5 <= angle < 112.5:
            direction = 'UP'
        elif 112.5 <= angle < 157.5:
            direction = 'UP-LEFT'
        elif angle >= 157.5 or angle < -157.5:
            direction = 'LEFT'
        elif -157.5 <= angle < -112.5:
            direction = 'DOWN-LEFT'
        elif -112.5 <= angle < -67.5:
            direction = 'DOWN'
        else:
            direction = 'DOWN-RIGHT'

    direction_counts[direction] += 1

print('DIRECTION SUMMARY')
print('------------------')
for direction, count in direction_counts.most_common():
    print(f'{direction}: {count}')


In [ ]:
# Combined vehicle analytics.
vehicle_data = {}
previous_position = {}
movement_data = defaultdict(lambda: [0.0, 0.0])
speed_data = defaultdict(list)

results = model.track(
    source=TRACKED_VIDEO,
    tracker='bytetrack.yaml',
    conf=0.25,
    stream=True,
    verbose=False
)

for result in results:
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    classes = result.boxes.cls.cpu().numpy().astype(int)

    for box, track_id, cls_id in zip(boxes, ids, classes):
        x1, y1, x2, y2 = box
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        if track_id in previous_position:
            old_x, old_y = previous_position[track_id]
            dx = cx - old_x
            dy = cy - old_y
            if abs(dx) > 1 or abs(dy) > 1:
                movement_data[track_id][0] += dx
                movement_data[track_id][1] += dy
                speed_data[track_id].append(math.sqrt(dx**2 + dy**2))

        previous_position[track_id] = (cx, cy)
        vehicle_data[track_id] = {'class': model.names[cls_id]}

for track_id, data in vehicle_data.items():
    dx, dy = movement_data[track_id]
    distance = math.sqrt(dx**2 + dy**2)

    if distance < 10:
        direction = 'STATIONARY'
    else:
        angle = math.degrees(math.atan2(-dy, dx))
        if -22.5 <= angle < 22.5:
            direction = 'RIGHT'
        elif 22.5 <= angle < 67.5:
            direction = 'UP-RIGHT'
        elif 67.5 <= angle < 112.5:
            direction = 'UP'
        elif 112.5 <= angle < 157.5:
            direction = 'UP-LEFT'
        elif angle >= 157.5 or angle < -157.5:
            direction = 'LEFT'
        elif -157.5 <= angle < -112.5:
            direction = 'DOWN-LEFT'
        elif -112.5 <= angle < -67.5:
            direction = 'DOWN'
        else:
            direction = 'DOWN-RIGHT'

    avg_speed = sum(speed_data[track_id]) / len(speed_data[track_id]) if speed_data[track_id] else 0
    vehicle_data[track_id]['direction'] = direction
    vehicle_data[track_id]['speed_px_frame'] = round(avg_speed, 2)

print('VEHICLE ANALYTICS COMPLETED')
print('---------------------------')
print('Vehicles:', len(vehicle_data))
for track_id, data in list(vehicle_data.items())[:10]:
    print(f"ID {track_id} | {data['class']} | {data['direction']} | {data['speed_px_frame']} px/frame")


In [ ]:
# Filter out very short/unstable tracks.
MIN_TRACK_FRAMES = 15
reliable_vehicles = {}

for track_id, data in vehicle_data.items():
    frames = len(speed_data[track_id])
    dx, dy = movement_data[track_id]
    total_distance = (dx**2 + dy**2) ** 0.5

    if frames >= MIN_TRACK_FRAMES:
        reliable_vehicles[track_id] = {
            'class': data['class'],
            'frames': frames,
            'total_movement': round(total_distance, 2),
            'direction': data['direction'],
            'speed_px_frame': data['speed_px_frame']
        }

print('RELIABLE VEHICLE TRACKS')
print('-----------------------')
print('Total track IDs   :', len(vehicle_data))
print('Reliable tracks   :', len(reliable_vehicles))
for track_id, data in list(reliable_vehicles.items())[:10]:
    print(f"ID {track_id} | {data['class']} | {data['frames']} frames | {data['direction']} | {data['speed_px_frame']} px/frame")


In [ ]:
# Final traffic analytics summary.
class_counts = Counter()
direction_counts = Counter()
speeds = []
moving = 0
stationary = 0

for data in reliable_vehicles.values():
    class_counts[data['class']] += 1
    direction_counts[data['direction']] += 1
    speeds.append(data['speed_px_frame'])
    if data['direction'] == 'STATIONARY':
        stationary += 1
    else:
        moving += 1

average_speed = sum(speeds) / len(speeds) if speeds else 0

print('TRAFFIC ANALYTICS SUMMARY')
print('=========================')
print('Reliable vehicles :', len(reliable_vehicles))
print('Moving vehicles   :', moving)
print('Stationary        :', stationary)
print('\nVehicle classes')
print('----------------')
for vehicle, count in class_counts.items():
    print(f'{vehicle}: {count}')
print('\nDirections')
print('----------------')
for direction, count in direction_counts.items():
    print(f'{direction}: {count}')
print('\nAverage speed')
print('----------------')
print(f'{average_speed:.2f} pixels/frame')
